# Human Detection menggunakan OpenCV HOG + Default People Detector
### Robby Azwan Saputra — 140810230008
### Pengolahan & Analisis Citra Digital

Implementasi human detection menggunakan **OpenCV built-in HOG descriptor** dengan **default people detector** (model SVM pretrained bawaan OpenCV) — tidak perlu training dataset.

Referensi: *Hands-On Image Processing with Python*, Chapter 8: Object Detection in Images


## 1. Import Library

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


## 2. Load Gambar

Ganti path gambar sesuai file yang ingin dideteksi.


In [ ]:
# Ganti path di sini sesuai gambar yang ingin ditest
IMAGE_PATH = "images/walk.jpg"

img = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

print("Resolusi:", img.shape[1], "×", img.shape[0], "px")


## 3. Inisialisasi HOG Descriptor + Default People Detector

OpenCV menyediakan model SVM pretrained untuk deteksi manusia — tinggal load, tidak perlu training.


In [ ]:
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

print("HOG Descriptor siap digunakan.")


## 4. Deteksi Raw (Tanpa Grouping)

Jalankan `detectMultiScale` dengan `finalThreshold=0` untuk melihat semua kandidat bounding box sebelum NMS.


In [ ]:
(found_bounding_boxes, weights) = hog.detectMultiScale(
    img,
    winStride=(4, 4),
    padding=(8, 8),
    scale=1.1,
    finalThreshold=0
)

print("Jumlah raw bounding boxes:", len(found_bounding_boxes))


## 5. Fungsi Compute IoU & Non-Maximum Suppression

Implementasi langsung dari buku untuk membuang bounding box yang redundant/tumpuk.


In [ ]:
def compute_iou(box, boxes, box_area, boxes_area):
    ys1 = np.maximum(box[0], boxes[:, 0])
    xs1 = np.maximum(box[1], boxes[:, 1])
    ys2 = np.minimum(box[2], boxes[:, 2])
    xs2 = np.minimum(box[3], boxes[:, 3])
    intersections = np.maximum(ys2 - ys1, 0) * np.maximum(xs2 - xs1, 0)
    unions = box_area + boxes_area - intersections
    ious = intersections / unions
    return ious


def non_max_suppression(boxes, scores, threshold):
    assert boxes.shape[0] == scores.shape[0]
    ys1, xs1, ys2, xs2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (ys2 - ys1) * (xs2 - xs1)
    scores_indexes = scores.argsort().tolist()
    boxes_keep_index = []

    while len(scores_indexes):
        index = scores_indexes.pop()
        boxes_keep_index.append(index)
        if not len(scores_indexes):
            break
        ious = compute_iou(
            boxes[index], boxes[scores_indexes],
            areas[index], areas[scores_indexes]
        )
        filtered_indexes = set((ious > threshold).nonzero()[0])
        scores_indexes = [v for (i, v) in enumerate(scores_indexes)
                          if i not in filtered_indexes]

    return np.array(boxes_keep_index)


print("Fungsi NMS siap.")


## 6. Terapkan NMS pada Hasil Deteksi


In [ ]:
# Konversi format box dari (x, y, w, h) ke (y1, x1, y2, x2)
boxes_for_nms = found_bounding_boxes.copy().astype(float)
boxes_for_nms[:, 2] = boxes_for_nms[:, 0] + boxes_for_nms[:, 2]  # x2
boxes_for_nms[:, 3] = boxes_for_nms[:, 1] + boxes_for_nms[:, 3]  # y2
boxes_for_nms = boxes_for_nms[:, [1, 0, 3, 2]]  # → (y1, x1, y2, x2)

box_indices = non_max_suppression(boxes_for_nms, weights.ravel(), threshold=0.2)
found_bounding_boxes = found_bounding_boxes[box_indices, :]

print("Boxes setelah NMS:", len(found_bounding_boxes))


## 7. Visualisasi Hasil Deteksi


In [ ]:
img_result = img.copy()

for (hx, hy, hw, hh) in found_bounding_boxes:
    cv2.rectangle(img_result, (hx, hy), (hx + hw, hy + hh), (0, 255, 0), 2)

img_result_rgb = cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_result_rgb)
plt.title(f"Detected Humans: {len(found_bounding_boxes)}")
plt.axis("off")
plt.show()


## 8. Alternatif: Meanshift Grouping

Selain NMS, buku juga menyebutkan `useMeanshiftGrouping=True` sebagai cara lain menangani box tumpuk.


In [ ]:
(ms_boxes, ms_weights) = hog.detectMultiScale(
    img,
    winStride=(4, 4),
    padding=(8, 8),
    scale=1.01,
    useMeanshiftGrouping=True
)

print("Boxes dengan meanshift grouping:", len(ms_boxes))

img_ms = img.copy()
for (hx, hy, hw, hh) in ms_boxes:
    cv2.rectangle(img_ms, (hx, hy), (hx + hw, hy + hh), (0, 0, 255), 2)

img_ms_rgb = cv2.cvtColor(img_ms, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_ms_rgb)
plt.title(f"Meanshift Grouping — Detected: {len(ms_boxes)}")
plt.axis("off")
plt.show()


## 9. Perbandingan: NMS vs Meanshift Grouping


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# NMS result
axes[0].imshow(img_result_rgb)
axes[0].set_title(f"NMS — {len(found_bounding_boxes)} deteksi", fontsize=13)
axes[0].axis("off")

# Meanshift result
axes[1].imshow(img_ms_rgb)
axes[1].set_title(f"Meanshift Grouping — {len(ms_boxes)} deteksi", fontsize=13)
axes[1].axis("off")

plt.suptitle("Perbandingan Metode Penanganan Box Tumpuk", fontsize=14)
plt.tight_layout()
plt.show()
